# Lab | Error Handling

## Exercise: Error Handling for Managing Customer Orders

The implementation of your code for managing customer orders assumes that the user will always enter a valid input. 

For example, we could modify the `initialize_inventory` function to include error handling.
   - If the user enters an invalid quantity (e.g., a negative value or a non-numeric value), display an error message and ask them to re-enter the quantity for that product.
   - Use a try-except block to handle the error and continue prompting the user until a valid quantity is entered.

```python
# Step 1: Define the function for initializing the inventory with error handling
def initialize_inventory(products):
    inventory = {}
    for product in products:
        valid_quantity = False
        while not valid_quantity:
            try:
                quantity = int(input(f"Enter the quantity of {product}s available: "))
                if quantity < 0:
                    raise ValueError("Invalid quantity! Please enter a non-negative value.")
                valid_quantity = True
            except ValueError as error:
                print(f"Error: {error}")
        inventory[product] = quantity
    return inventory

# Or, in another way:

def initialize_inventory(products):
    inventory = {}
    for product in products:
        valid_input = False
        while not valid_input:
            try:
                quantity = int(input(f"Enter the quantity of {product}s available: "))
                if quantity >= 0:
                    inventory[product] = quantity
                    valid_input = True
                else:
                    print("Quantity cannot be negative. Please enter a valid quantity.")
            except ValueError:
                print("Invalid input. Please enter a valid quantity.")
    return inventory
```

Let's enhance your code by implementing error handling to handle invalid inputs.

Follow the steps below to complete the exercise:

2. Modify the `calculate_total_price` function to include error handling.
   - If the user enters an invalid price (e.g., a negative value or a non-numeric value), display an error message and ask them to re-enter the price for that product.
   - Use a try-except block to handle the error and continue prompting the user until a valid price is entered.

3. Modify the `get_customer_orders` function to include error handling.
   - If the user enters an invalid number of orders (e.g., a negative value or a non-numeric value), display an error message and ask them to re-enter the number of orders.
   - If the user enters an invalid product name (e.g., a product name that is not in the inventory), or that doesn't have stock available, display an error message and ask them to re-enter the product name. *Hint: you will need to pass inventory as a parameter*
   - Use a try-except block to handle the error and continue prompting the user until a valid product name is entered.

4. Test your code by running the program and deliberately entering invalid quantities and product names. Make sure the error handling mechanism works as expected.


## Resolución

Parto de la versión del lab de *comprehension* y añado control de errores a las tres funciones que reciben datos del usuario: `initialize_inventory`, `get_customer_orders` y `calculate_total_price`. Las demás no leen nada del teclado, así que no las toco.

La idea en todas es la misma:

1. Un bucle `while` que se repite hasta que el dato sea válido.
2. Dentro, un bloque `try` con la conversión (`int()` o `float()`), que lanza `ValueError` si el texto no es un número.
3. Si el número no tiene sentido (negativo, producto inexistente...), lo lanzo yo con `raise ValueError("mensaje")`. Así todos los errores acaban en el mismo `except` y se tratan igual.
4. El `except ValueError as error` muestra el mensaje y el bucle vuelve a preguntar.

**Por qué `ValueError` y no un `except:` a secas:** un `except` sin tipo atrapa cualquier cosa, incluso errores de programación o que el usuario pulse Ctrl+C, y los esconde. Capturar solo `ValueError` trata el caso previsto (dato inválido) y deja ver los demás.

In [1]:
products = ["t-shirt", "mug", "hat", "book", "keychain"]

### Paso 1. `initialize_inventory` con control de errores

Uso la primera versión que propone el enunciado.

**Qué hace:** para cada producto pide la cantidad hasta que sea un entero mayor o igual que 0.

**Por qué aquí dejo el `for` en vez de la comprensión del lab anterior:** dentro de una comprensión no se puede meter un `while` ni un `try/except`. Cuando la lógica de cada elemento crece, el bucle normal es más legible.

**Por qué `raise` dentro del `try`:** `int("-3")` funciona sin error, así que Python no se queja de un negativo. Lo convierto en error a mano para que caiga en el mismo `except` que `int("abc")` y el mensaje salga por el mismo sitio.

In [2]:
def initialize_inventory(products):
    inventory = {}
    for product in products:
        valid_quantity = False
        while not valid_quantity:
            try:
                quantity = int(input(f"Enter the quantity of {product}s available: "))
                if quantity < 0:
                    raise ValueError("Invalid quantity! Please enter a non-negative value.")
                valid_quantity = True
            except ValueError as error:
                print(f"Error: {error}")
        inventory[product] = quantity
    return inventory

### Paso 2. `calculate_total_price` con control de errores

**Qué hace:** para cada producto pide el precio hasta que sea un número mayor o igual que 0, y va acumulando el total.

**Por qué `float` y no `int`:** un precio puede tener decimales (`9.99`).

**Por qué `while True` con `break`:** es otra forma de escribir el bucle de validación, sin variable bandera. Sale del bucle en cuanto el precio es válido. Uso las dos formas en el lab para practicar ambas; hacen lo mismo.

**Por qué aquí no uso `raise`:** en el Paso 1 el mensaje de un texto no numérico es el de Python (`invalid literal for int() with base 10: 'abc'`), que es poco claro para un usuario. Aquí separo los dos casos: el `try/except` solo envuelve la conversión y pone un mensaje propio; el negativo se comprueba después con un `if` normal. `continue` vuelve al principio del `while` para preguntar otra vez. Las dos formas son válidas; esta da mensajes más claros y la del Paso 1 practica `raise`.

In [3]:
def calculate_total_price(customer_orders):
    total_price = 0
    for product in customer_orders:
        while True:
            try:
                price = float(input(f"Enter the price of {product}: "))
            except ValueError:
                print("Error: the price must be a number.")
                continue
            if price < 0:
                print("Error: the price cannot be negative.")
                continue
            break
        total_price += price
    return total_price

### Paso 3. `get_customer_orders` con control de errores

Tiene dos validaciones distintas, así que las separo:

**3a. Número de pedidos.** Debe ser un entero mayor o igual que 0. Mismo patrón que en el Paso 1.

**3b. Nombre de cada producto.** Debe existir en el inventario **y** tener stock (`inventory[product] > 0`). Por eso ahora la función recibe `inventory` como parámetro, como indica la pista del enunciado.

**Por qué comprobar `product not in inventory` antes que el stock:** si el producto no existe, `inventory[product]` daría `KeyError`. El `or` de Python deja de evaluar en cuanto la primera condición es verdadera, así que nunca llega a buscar un producto inexistente.

**Por qué rechazo también un producto repetido:** el pedido es un `set`, así que un producto repetido no se añadiría y acabaríamos con menos pedidos de los indicados. Avisar al usuario es más claro que perder el pedido en silencio.

In [4]:
def get_customer_orders(inventory):
    # 3a. Número de pedidos
    while True:
        try:
            number_of_orders = int(input("Enter the number of customer orders: "))
            if number_of_orders < 0:
                raise ValueError("the number of orders cannot be negative.")
            break
        except ValueError as error:
            print(f"Error: {error}")

    # 3b. Nombre de cada producto
    customer_orders = set()
    for _ in range(number_of_orders):
        while True:
            try:
                product = input("Enter the name of a product that a customer wants to order: ").strip().lower()
                if product not in inventory:
                    raise ValueError(f"'{product}' is not in the inventory.")
                if inventory[product] <= 0:
                    raise ValueError(f"'{product}' is out of stock.")
                if product in customer_orders:
                    raise ValueError(f"'{product}' is already in the order.")
                customer_orders.add(product)
                break
            except ValueError as error:
                print(f"Error: {error}")
    return customer_orders

### Paso 4. Funciones sin entrada del usuario

Son las del lab de comprehension, sin cambios: no leen del teclado, así que no pueden recibir datos inválidos del usuario.

In [5]:
def update_inventory(customer_orders, inventory):
    inventory = {product: (quantity - 1 if product in customer_orders else quantity)
                 for product, quantity in inventory.items()}
    inventory = {product: quantity for product, quantity in inventory.items() if quantity > 0}
    return inventory


def calculate_order_statistics(customer_orders, products):
    total_products_ordered = len(customer_orders)
    percentage_ordered = total_products_ordered / len(products) * 100
    return total_products_ordered, percentage_ordered


def print_order_statistics(order_statistics):
    total_products_ordered, percentage_ordered = order_statistics
    print("\nOrder Statistics:")
    print(f"Total Products Ordered: {total_products_ordered}")
    print(f"Percentage of Unique Products Ordered: {percentage_ordered}")


def print_updated_inventory(inventory):
    print("\nUpdated Inventory:")
    print("\n".join(f"{product}: {quantity}" for product, quantity in inventory.items()))

### Paso 5. Programa principal y prueba con datos inválidos

El punto 4 del enunciado pide probar el programa **introduciendo errores a propósito**. En la ejecución guardada abajo he usado estos datos:

| Pregunta | Escrito | Resultado esperado |
|---|---|---|
| cantidad de t-shirts | `abc`, luego `5` | error por no ser número, luego acepta |
| cantidad de mugs | `-2`, luego `4` | error por negativo, luego acepta |
| cantidades de hats, books, keychains | `3`, `2`, `0` | acepta (0 es válido) |
| número de pedidos | `dos`, luego `2` | error, luego acepta |
| producto 1 | `banana`, luego `keychain`, luego `hat` | no existe; sin stock; acepta |
| producto 2 | `hat`, luego `book` | repetido; acepta |
| precio de cada producto | `diez`, `-10`, `10` y `5` | error de texto, error de negativo, acepta |

Cuando lo ejecutes tú, prueba tus propios errores para ver que todos los mensajes salen bien.

In [6]:
inventory = initialize_inventory(products)
customer_orders = get_customer_orders(inventory)

order_statistics = calculate_order_statistics(customer_orders, products)
print_order_statistics(order_statistics)

inventory = update_inventory(customer_orders, inventory)
print_updated_inventory(inventory)

total_price = calculate_total_price(customer_orders)
print(f"Total Price: {total_price}")

Enter the quantity of t-shirts available: abc
Error: invalid literal for int() with base 10: 'abc'
Enter the quantity of t-shirts available: 5
Enter the quantity of mugs available: -2
Error: Invalid quantity! Please enter a non-negative value.
Enter the quantity of mugs available: 4
Enter the quantity of hats available: 3
Enter the quantity of books available: 2
Enter the quantity of keychains available: 0
Enter the number of customer orders: dos
Error: invalid literal for int() with base 10: 'dos'
Enter the number of customer orders: 2
Enter the name of a product that a customer wants to order: banana
Error: 'banana' is not in the inventory.
Enter the name of a product that a customer wants to order: keychain
Error: 'keychain' is out of stock.
Enter the name of a product that a customer wants to order: hat
Enter the name of a product that a customer wants to order: hat
Error: 'hat' is already in the order.
Enter the name of a product that a customer wants to order: book

Order Statist